# All of Us CDRv9 — Stage 0 Multimodal Feasibility\n\nRun this notebook **inside the All of Us Researcher Workbench 2.0 Controlled Tier**.\n\nIt checks Fitbit, EHR, WGS and proteomics overlap, derives first-pass wearable QC, and returns a GO / CONDITIONAL / PIVOT decision for the thesis design. It writes **aggregate counts only**.\n\n**Before running:** attach C2025Q4R6 and the CDRv9 genomics/multi-omics resources, then run the official Getting Started with Verily Workbench setup notebook so WORKSPACE_CDR is defined.

In [ ]:
# All of Us CDRv9 Stage 0 — ONE RUN CELL
# Run inside an All of Us Researcher Workbench 2.0 Controlled Tier Jupyter workspace.
# Before this cell, run the official "Getting Started with Verily Workbench" notebook once
# so WORKSPACE_CDR and data-collection resources are available.

import os, sys, json
from pathlib import Path
import pandas as pd

# --- CONFIG ---------------------------------------------------------------
WORKSPACE_CDR_OVERRIDE = None

# If auto-discovery does not find these resources, paste the path from the
# Controlled Tier Data Dictionary / workspace resource into the variables below.
WGS_SAMPLE_MANIFEST = None
PROTEOMICS_SAMPLE_RESOURCE = None

WGS_ID_COLUMN = None
PROTEOMICS_ID_COLUMN = None

MIN_SLEEP_DAYS = 21
PREFERRED_SLEEP_DAYS = 30
# -------------------------------------------------------------------------

candidate_helpers = [
    Path("../research/digital_health_omics/allofus_stage0.py"),
    Path("research/digital_health_omics/allofus_stage0.py"),
]
helper_path = next((p.resolve() for p in candidate_helpers if p.exists()), None)
if helper_path is None:
    raise FileNotFoundError(
        "allofus_stage0.py not found. Clone the Codex repo or run this notebook "
        "from notebooks/ inside the repo."
    )

sys.path.insert(0, str(helper_path.parent))
from allofus_stage0 import (
    FITBIT_TABLES, EHR_TABLES,
    get_cdr_ref, get_bq_client, available_tables, distinct_ids_sql,
    fetch_ids, environment_resource_hints,
    infer_wgs_manifest_path, infer_proteomics_path, load_participant_ids,
    sleep_daily_aggregate_sql, activity_daily_aggregate_sql,
    make_qc_summary, filter_qc_ids, stage0_summary, proposal_decision,
)

cdr = get_cdr_ref(WORKSPACE_CDR_OVERRIDE)
client = get_bq_client()

print("="*78)
print("All of Us CDRv9 Stage 0")
print("="*78)
print("WORKSPACE_CDR:", cdr)

# 1) BigQuery modality discovery
fitbit_tables = available_tables(client, cdr, FITBIT_TABLES)
ehr_tables = available_tables(client, cdr, EHR_TABLES)
print("\nFitbit tables:", fitbit_tables)
print("EHR tables:", ehr_tables)

if not fitbit_tables:
    raise RuntimeError("No Fitbit tables found. Confirm that CDRv9 Controlled Tier is attached.")

fitbit_sql = distinct_ids_sql(cdr, [
    t for t in fitbit_tables
    if t in {"sleep_daily_summary_ext","sleep_daily_summary","activity_summary","heart_rate_summary"}
])
ehr_sql = distinct_ids_sql(cdr, ehr_tables)

print("\nCounting Fitbit/EHR participants...")
fitbit_ids = fetch_ids(client, fitbit_sql)
ehr_ids = fetch_ids(client, ehr_sql)
print(f"Fitbit: {len(fitbit_ids):,}")
print(f"EHR:    {len(ehr_ids):,}")

# 2) Genomics / proteomics resource discovery
print("\nRelevant workspace environment resources:")
display(environment_resource_hints())

wgs_path = WGS_SAMPLE_MANIFEST or infer_wgs_manifest_path()
prot_path = PROTEOMICS_SAMPLE_RESOURCE or infer_proteomics_path()

print("\nResolved WGS manifest:", wgs_path or "NOT FOUND")
print("Resolved proteomics resource:", prot_path or "NOT FOUND")

wgs_ids = set()
proteomics_ids = set()

if wgs_path:
    try:
        wgs_ids = load_participant_ids(wgs_path, WGS_ID_COLUMN)
        print(f"WGS manifest IDs: {len(wgs_ids):,}")
    except Exception as e:
        print("WGS ID load warning:", e)

if prot_path:
    try:
        proteomics_ids = load_participant_ids(prot_path, PROTEOMICS_ID_COLUMN)
        print(f"Proteomics IDs: {len(proteomics_ids):,}")
    except Exception as e:
        print("Proteomics ID load warning:", e)

if not wgs_ids:
    print(
        "\n[WGS ACTION] Add/select the CDRv9 genomics resource in Controlled Tier. "
        "If auto-discovery still fails, set WGS_SAMPLE_MANIFEST to a manifest "
        "containing person_id."
    )
if not proteomics_ids:
    print(
        "\n[PROTEOMICS ACTION] Add/select the CDRv9 proteomics resource. "
        "Set PROTEOMICS_SAMPLE_RESOURCE to a sample metadata or replicate-removed "
        "TSV/CSV/Parquet exposing ResearchID. CDRv9 proteomics has 9,969 participants."
    )

# 3) Wearable QC / first digital phenotypes
print("\nBuilding participant-level sleep/activity summaries in BigQuery...")
sleep_sql, sleep_meta = sleep_daily_aggregate_sql(client, cdr)
sleep_df = client.query(sleep_sql).to_dataframe()
print("Sleep source mapping:", json.dumps(sleep_meta, indent=2))

try:
    activity_sql, activity_meta = activity_daily_aggregate_sql(client, cdr)
    activity_df = client.query(activity_sql).to_dataframe()
    print("Activity source mapping:", json.dumps(activity_meta, indent=2))
except Exception as e:
    print("Activity summary warning:", e)
    activity_df = pd.DataFrame()

qc = make_qc_summary(sleep_df, activity_df)
qc_ids = filter_qc_ids(qc, MIN_SLEEP_DAYS)

print("\nWearable QC:")
print(f"Participants with >= {MIN_SLEEP_DAYS} sleep days: {len(qc_ids):,}")
print(f"Participants with >= {PREFERRED_SLEEP_DAYS} sleep days: "
      f"{int(qc['n_sleep_days'].fillna(0).ge(PREFERRED_SLEEP_DAYS).sum()):,}")

# 4) Multimodal overlap
summary = stage0_summary(
    fitbit_ids=fitbit_ids,
    ehr_ids=ehr_ids,
    wgs_ids=wgs_ids,
    proteomics_ids=proteomics_ids,
    qc_ids=qc_ids if (wgs_ids and proteomics_ids) else None,
)

print("\n" + "="*78)
print("MODALITY OVERLAP")
print("="*78)
display(summary)

final_n = None
if wgs_ids and proteomics_ids:
    final_ids = fitbit_ids & ehr_ids & wgs_ids & proteomics_ids & qc_ids
    final_n = len(final_ids)

decision, reason = proposal_decision(final_n)
print("\n" + "="*78)
print("THESIS DECISION:", decision)
print(reason)
print("="*78)

# 5) Stage-0 digital phenotype preview — aggregate only
preview_cols = [
    c for c in [
        "person_id", "n_sleep_days", "mean_sleep_hours", "sd_sleep_hours",
        "mean_sleep_efficiency", "n_activity_days", "mean_steps", "sd_steps",
        "n_days_steps_ge_100", "sleep_qc_21d", "sleep_qc_30d"
    ] if c in qc.columns
]
print("\nSleep/activity phenotype columns generated:")
print(preview_cols)
display(qc[preview_cols].describe(include="all").T)

# Save only aggregate Stage-0 outputs.
out = Path("aou_stage0_outputs")
out.mkdir(exist_ok=True)
summary.to_csv(out / "modality_overlap_counts.csv", index=False)
pd.DataFrame([{
    "decision": decision,
    "reason": reason,
    "min_sleep_days": MIN_SLEEP_DAYS,
    "preferred_sleep_days": PREFERRED_SLEEP_DAYS,
}]).to_csv(out / "stage0_decision.csv", index=False)
print("\nSaved aggregate-only outputs to:", out.resolve())
